# White-box adversarial training

This notebook trains:
1. Clean ResNet-18 layer3 wrapper model,
2. Input-level PGD adversarially trained model,
3. Latent-level PGD adversarially trained model,
4. Combined input + latent PGD adversarially trained model.

In [1]:
# Imports
from xml.parsers.expat import model

import os
import csv
import pickle

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.functional as F

import utils

In [2]:
# Configuration
# In a .py script, BASE_DIR was os.path.dirname(os.path.abspath(__file__)).
# In a notebook, use the current working directory or set this manually.
BASE_DIR = os.getcwd()

images, labels = utils.readTrafficSigns_train(BASE_DIR + "/GTSRB/Training/Final_Training/Images")


CACHE_PATH = os.path.join(BASE_DIR, "gtsrb_cache.pkl")
with open(CACHE_PATH, "wb") as f:
    pickle.dump((images, labels), f)    

print(f"Cached data to {CACHE_PATH}")



Cached data to /mnt/c/Users/topyo/Documents/ML_Project_2026/Whitebox/gtsrb_cache.pkl


In [6]:
num_classes = 43
batch_size_train = 64
num_workers = 2
pin_memory = True

clean_epochs = 20
adv_epochs = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load cached training data
print("Reading training data...")

with open(CACHE_PATH, "rb") as f:
    trainImages, trainLabels = pickle.load(f)

train_dataset = utils.GTSRBDataset(trainImages, trainLabels, target_size=224)
print(f"Total training samples: {len(train_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

Using device: cuda
Reading training data...
Total training samples: 39209


In [7]:
# Shared training objects
criterion = nn.CrossEntropyLoss()

def make_model():
    model = utils.ResNet18_L3(num_classes=num_classes)
    return model.to(device)

def make_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

## 1. Clean training

In [8]:
# Clean training
model = make_model()
optimizer = make_optimizer(model)

for epoch in range(clean_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(utils.normalize_batch(images))
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Clean Epoch {epoch + 1}/{clean_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

clean_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_clean.pth")
torch.save(model.state_dict(), clean_checkpoint_path)
print(f"Clean training complete. Model saved as {clean_checkpoint_path}")

Clean Epoch 1/20, Loss: 0.0053, Accuracy: 94.74%
Clean Epoch 2/20, Loss: 0.0027, Accuracy: 99.93%
Clean Epoch 3/20, Loss: 0.0025, Accuracy: 99.94%
Clean Epoch 4/20, Loss: 0.0057, Accuracy: 99.92%
Clean Epoch 5/20, Loss: 0.0265, Accuracy: 99.87%
Clean Epoch 6/20, Loss: 0.0013, Accuracy: 99.91%
Clean Epoch 7/20, Loss: 0.0005, Accuracy: 99.97%
Clean Epoch 8/20, Loss: 0.0003, Accuracy: 100.00%
Clean Epoch 9/20, Loss: 0.0003, Accuracy: 100.00%
Clean Epoch 10/20, Loss: 0.0001, Accuracy: 100.00%
Clean Epoch 11/20, Loss: 0.2572, Accuracy: 99.99%
Clean Epoch 12/20, Loss: 0.0008, Accuracy: 99.60%
Clean Epoch 13/20, Loss: 0.0018, Accuracy: 99.94%
Clean Epoch 14/20, Loss: 0.0010, Accuracy: 99.83%
Clean Epoch 15/20, Loss: 0.0005, Accuracy: 99.99%
Clean Epoch 16/20, Loss: 0.0061, Accuracy: 99.96%
Clean Epoch 17/20, Loss: 0.0046, Accuracy: 100.00%
Clean Epoch 18/20, Loss: 0.0002, Accuracy: 100.00%
Clean Epoch 19/20, Loss: 0.0003, Accuracy: 100.00%
Clean Epoch 20/20, Loss: 0.0039, Accuracy: 99.69%
Cle

## 2. Input-level PGD adversarial training

In [9]:
# Input-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

lambda_clean = 1 / 2
lambda_input_adv = 1 / 2

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        adv_input_images = utils.generate_adversarial_batch(
            model=model,
            images=images,
            labels=labels,
            attack="PGD",
            device=device,
            epsilon=8 / 255.0,
            alpha=2 / 255.0,
            num_steps=10,
            criterion=criterion,
        )

        adv_input_outputs = model(utils.normalize_batch(adv_input_images))
        adv_input_loss = criterion(adv_input_outputs, labels)

        loss = (lambda_clean * clean_loss) + (lambda_input_adv * adv_input_loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_input_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Input-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

input_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_input_adv.pth")
torch.save(model.state_dict(), input_adv_checkpoint_path)
print(f"Saved input-PGD robust model to {input_adv_checkpoint_path}")

Input-PGD Epoch 1/15, Loss: 0.5029, Accuracy: 59.02%
Input-PGD Epoch 2/15, Loss: 0.5457, Accuracy: 72.76%
Input-PGD Epoch 3/15, Loss: 0.3399, Accuracy: 76.61%
Input-PGD Epoch 4/15, Loss: 0.4647, Accuracy: 78.62%
Input-PGD Epoch 5/15, Loss: 0.3887, Accuracy: 79.78%
Input-PGD Epoch 6/15, Loss: 0.3003, Accuracy: 81.01%
Input-PGD Epoch 7/15, Loss: 0.2049, Accuracy: 81.31%
Input-PGD Epoch 8/15, Loss: 0.3185, Accuracy: 82.91%
Input-PGD Epoch 9/15, Loss: 0.2601, Accuracy: 83.46%
Input-PGD Epoch 10/15, Loss: 0.2827, Accuracy: 84.17%
Input-PGD Epoch 11/15, Loss: 0.2351, Accuracy: 84.29%
Input-PGD Epoch 12/15, Loss: 0.4902, Accuracy: 85.01%
Input-PGD Epoch 13/15, Loss: 0.4187, Accuracy: 85.64%
Input-PGD Epoch 14/15, Loss: 0.3213, Accuracy: 85.86%
Input-PGD Epoch 15/15, Loss: 0.2490, Accuracy: 86.41%
Saved input-PGD robust model to /mnt/c/Users/topyo/Documents/ML_Project_2026/Whitebox/resnet18_gtsrb_input_adv.pth


## 3. Latent-level PGD adversarial training

In [10]:
# Latent-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

epsilon_latent = 0.2
lambda_latent_adv = 1 / 2
lambda_clean = 1 / 2

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        z_adv = utils.pgd_attack_latent_l3(
            model=model,
            image=images,
            label=labels,
            epsilon=epsilon_latent,
            alpha=epsilon_latent / 4.0,
            num_steps=5,
            criterion=criterion,
            device=device,
        )

        adv_latent_outputs = model.decode(z_adv)
        adv_latent_loss = criterion(adv_latent_outputs, labels)

        loss = (lambda_clean * clean_loss) + (lambda_latent_adv * adv_latent_loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_latent_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Latent-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

latent_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_latent_adv.pth")
torch.save(model.state_dict(), latent_adv_checkpoint_path)
print(f"Saved latent-PGD robust model to {latent_adv_checkpoint_path}")

Latent-PGD Epoch 1/15, Loss: 0.6664, Accuracy: 55.16%
Latent-PGD Epoch 2/15, Loss: 0.4227, Accuracy: 78.25%
Latent-PGD Epoch 3/15, Loss: 0.2812, Accuracy: 87.98%
Latent-PGD Epoch 4/15, Loss: 0.0975, Accuracy: 94.11%
Latent-PGD Epoch 5/15, Loss: 0.1721, Accuracy: 92.06%
Latent-PGD Epoch 6/15, Loss: 0.0307, Accuracy: 95.06%
Latent-PGD Epoch 7/15, Loss: 0.1303, Accuracy: 96.41%
Latent-PGD Epoch 8/15, Loss: 0.1450, Accuracy: 92.79%
Latent-PGD Epoch 9/15, Loss: 0.0804, Accuracy: 94.58%
Latent-PGD Epoch 10/15, Loss: 0.0411, Accuracy: 97.09%
Latent-PGD Epoch 11/15, Loss: 0.1594, Accuracy: 96.95%
Latent-PGD Epoch 12/15, Loss: 0.0983, Accuracy: 96.53%
Latent-PGD Epoch 13/15, Loss: 0.2153, Accuracy: 96.23%
Latent-PGD Epoch 14/15, Loss: 0.1878, Accuracy: 95.02%
Latent-PGD Epoch 15/15, Loss: 0.2385, Accuracy: 91.30%
Saved latent-PGD robust model to /mnt/c/Users/topyo/Documents/ML_Project_2026/Whitebox/resnet18_gtsrb_latent_adv.pth


## 4. Combined input-level PGD + latent-level PGD adversarial training

In [11]:
# Combined input-level PGD and latent-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

epsilon_latent = 0.2
lambda_latent_adv = 1 / 3
lambda_clean = 1 / 3
lambda_input_adv = 1 / 3

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        adv_input_images = utils.generate_adversarial_batch(
            model=model,
            images=images,
            labels=labels,
            attack="PGD",
            device=device,
            epsilon=8 / 255.0,
            alpha=2 / 255.0,
            num_steps=10,
            criterion=criterion,
        )

        adv_input_outputs = model(utils.normalize_batch(adv_input_images))
        adv_input_loss = criterion(adv_input_outputs, labels)

        z_adv = utils.pgd_attack_latent_l3(
            model=model,
            image=images,
            label=labels,
            epsilon=epsilon_latent,
            alpha=epsilon_latent / 4.0,
            num_steps=5,
            criterion=criterion,
            device=device,
        )

        adv_latent_outputs = model.decode(z_adv)
        adv_latent_loss = criterion(adv_latent_outputs, labels)

        loss = (
            (lambda_clean * clean_loss)
            + (lambda_input_adv * adv_input_loss)
            + (lambda_latent_adv * adv_latent_loss)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_input_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Input+Latent-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

input_latent_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_input_latent_adv.pth")
torch.save(model.state_dict(), input_latent_adv_checkpoint_path)
print(f"Saved input-latent-PGD robust model to {input_latent_adv_checkpoint_path}")

Input+Latent-PGD Epoch 1/15, Loss: 1.1369, Accuracy: 57.83%
Input+Latent-PGD Epoch 2/15, Loss: 0.9454, Accuracy: 71.60%
Input+Latent-PGD Epoch 3/15, Loss: 0.8774, Accuracy: 75.11%
Input+Latent-PGD Epoch 4/15, Loss: 0.8164, Accuracy: 77.02%
Input+Latent-PGD Epoch 5/15, Loss: 0.5738, Accuracy: 78.32%
Input+Latent-PGD Epoch 6/15, Loss: 0.4211, Accuracy: 79.53%
Input+Latent-PGD Epoch 7/15, Loss: 0.7916, Accuracy: 80.36%
Input+Latent-PGD Epoch 8/15, Loss: 0.7085, Accuracy: 81.17%
Input+Latent-PGD Epoch 9/15, Loss: 0.4869, Accuracy: 81.63%
Input+Latent-PGD Epoch 10/15, Loss: 0.4926, Accuracy: 82.36%
Input+Latent-PGD Epoch 11/15, Loss: 0.5063, Accuracy: 82.82%
Input+Latent-PGD Epoch 12/15, Loss: 0.4874, Accuracy: 83.42%
Input+Latent-PGD Epoch 13/15, Loss: 0.3221, Accuracy: 83.91%
Input+Latent-PGD Epoch 14/15, Loss: 0.4029, Accuracy: 84.26%
Input+Latent-PGD Epoch 15/15, Loss: 0.5572, Accuracy: 84.52%
Saved input-latent-PGD robust model to /mnt/c/Users/topyo/Documents/ML_Project_2026/Whitebox/r